In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip3 install torch torchvision torchaudio

In [3]:
!pip install transformers

* `AutoTokenizer`: This class provides an easy way to load a pre-trained tokenizer. Tokenizers are responsible for converting input text into token IDs that the model can understand.

* `AutoModelForSequenceClassification`: This class is used to load a pre-trained model specifically designed for sequence classification tasks, such as sentiment analysis.

In [4]:
import torch
from transformers import AutoModelForSequenceClassification, AutoConfig
#from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, Trainer
from transformers import BertTokenizer, BertForSequenceClassification, BertConfig
import numpy 
import numpy as np
from scipy.special import softmax
import tensorflow as tf

2026-04-18 22:24:12.139060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776551052.163617     877 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776551052.170734     877 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776551052.189838     877 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776551052.189871     877 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776551052.189873     877 computation_placer.cc:177] computation placer alr

* `AutoTokenizer.from_pretrained`: This method loads a tokenizer from a pre-trained model specified by the model name or path.

* `'nlptown/bert-base-multilingual-uncased-sentiment'`: This is the identifier for a pre-trained BERT model provided by the nlptown organization on Hugging Face's model hub. This specific model has been fine-tuned for sentiment analysis across multiple languages and uses uncased text (meaning it does not distinguish between uppercase and lowercase letters).

* `AutoModelForSequenceClassification.from_pretrained`: This method loads a model for sequence classification from a pre-trained model specified by the model name or path.

* `'nlptown/bert-base-multilingual-uncased-sentiment'`: This is the same model identifier used for loading the tokenizer. It ensures that the tokenizer and the model are compatible with each other. The model is designed to classify sequences (text inputs) into different sentiment categories.

In [5]:
# load tokenizer and model, create trainer
MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
trainer = Trainer(model=model)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Apply the `sentiment_score` function to each review, truncating it to the first 512 tokens as BERT models have a maximum input length of 512 tokens, and stores the resulting sentiment scores in a new column named ‘sentiment’.

# Sentiment Scores

* **Very Negative (Score: 1)** Indicates strong negative sentiment. The review expresses a high level of dissatisfaction or negative experience.

* **Negative (Score: 2)** Indicates a negative sentiment. The review reflects dissatisfaction or an unpleasant experience but is not as strongly negative as a score of 1.

* **Neutral (Score: 3)** Indicates a neutral sentiment. The review does not lean strongly towards either positive or negative. It might reflect an average or mixed experience.

* **Positive (Score: 4)** Indicates a positive sentiment. The review expresses satisfaction or a good experience, though it is not as strongly positive as a score of 5.

* **Very Positive (Score: 5)** Indicates strong positive sentiment. The review expresses a high level of satisfaction or an excellent experience.

In [6]:
def sentiment_labels(text):
    encoded_input = tokenizer(text, padding=True,truncation=True,max_length=512,return_tensors='pt')
    output = model(**encoded_input)
    # Use .squeeze() to remove the batch dimension safely
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    ranking = np.argsort(scores)
    ranking = ranking[::-1]
    return config.id2label[ranking[0]]

In [7]:
data = [
        "Dr Priya Patel in the respiratory clinic was absolutely wonderful. She took time to explain my COPD management plan and made sure I understood every step. The nurse on reception, Karen Thompson, was also very welcoming.",
        "The parking at 45 Albert Street is very limited. I had to walk from the Westfield car park which is difficult at my age.",
        "I had my appointment on 15 January 2026 at 2pm and wasn't seen until after 3:15pm. Dr James Richardson seemed rushed and didn't listen to my concerns about the medication side effects. I called the clinic at 07 3456 7890 twice before my appointment to ask about preparation and nobody answered. My wife Angela Chen had a much better experience at the Chermside Day Surgery last month.",
        "The physiotherapist Michael was great. He gave me a detailed home exercise program after my knee surgery and followed up via email at susan.obrien82@gmail.com to check on my progress. Very impressed with that level of care.",
        "The online booking system could be easier to use. I ended up calling reception to book because I couldn't figure out the Zedoc portal. I was referred by Dr Nguyen at the Ipswich Hospital and the transition was smooth.",
        "Nurse Rebecca Taylor in the diabetes clinic was exceptional. She spent over 40 minutes with me going through my HbA1c results and adjusting my insulin plan. She also coordinated with my GP Dr Samantha Lee at the Toowoomba Medical Practice to ensure consistent care.",
        "Dr Patel is always thorough and patient. She remembers details from previous visits which makes me feel valued.",
        "I'm 81 years old and find the forms quite difficult to fill in. My daughter Sarah Fitzpatrick usually helps me but she wasn't available this time. Could you offer some assistance at the front desk for elderly patients? Also my Medicare number is 2345 67890 1 and I think there was a billing error on my last visit on 28 January 2026.",
        "The midwifery team, especially Lisa and Jenny, were amazing throughout my pregnancy. They made me feel safe and supported. The antenatal classes at the centre on George Street were also excellent.",
        "It would be nice to have later appointment slots. As a working mum I find it hard to attend before 4pm. My employer at Bright Horizons Childcare is not always flexible with time off. I recommended this clinic to my sister-in-law Patricia Nakamura who is expecting in July.",
        "The new blood collection nurse, Daniel Kim, was very skilled. Best blood draw I've had - barely felt it. He mentioned he previously worked at the Royal Brisbane and Women's Hospital.",
        "The results portal is confusing. I had to call Dr Richardson's office at 07 3456 7891 to get my pathology explained because I couldn't understand the online report.",
        "Dr Patel and Nurse Rebecca were both excellent. They were respectful of my cultural needs and ensured a female practitioner was available for my examination. This was very important to me.",
        "The interpreter service was not available on 4 March 2026 when my mother attended. She speaks Arabic and struggled to communicate her symptoms. Please ensure interpreters are booked in advance. My mother Zahra Al-Rashid (patient ID PAT-91603) would like to provide feedback separately. Can someone contact her at fatima.alrashid@outlook.com to arrange an Arabic-language survey?",
        "Outstanding mental health support from psychologist Dr Amanda Clarke. She helped me develop coping strategies after my workplace incident at BHP Mitsubishi Alliance in Mount Isa last year. The telehealth option made it possible to continue sessions when I was FIFO.",
        "The mental health waiting list is too long. I was referred on 15 November 2025 and didn't get my first appointment until 8 January 2026. That's nearly 8 weeks.",
        "The wound care nurse Jacinta was thorough and gentle. She explained the healing process for my post-surgical wound clearly and gave me written instructions to take home.",
        "I received a reminder SMS from Zedoc to complete this survey but the link didn't work on my Samsung phone. I had to use my husband David Tran's iPhone instead. The SMS came from number 0437 123 456. I was transferred from Logan Hospital after my surgery there on 2 March 2026. The handover between hospitals could have been smoother - my medication list wasn't updated correctly.",
        "The entire cardiology team is first-rate. Dr Richardson personally called me at home on 0478 234 567 to discuss my stress test results. That kind of proactive communication is rare and very reassuring.",
        "The car park needs more disabled bays. I have a temporary mobility permit after my cardiac rehab and struggled to find a spot on 20 March 2026. My cardiologist at the Prince Charles Hospital, Dr Andrew Walsh, also coordinates with Dr Richardson here which gives me great confidence in my care plan."
       ]

In [8]:
for i, text in enumerate(data):
    print(f" * sentece: {i + 1}: {text}")
    print(">>>", sentiment_labels(text), "\n")

 * sentece: 1: Dr Priya Patel in the respiratory clinic was absolutely wonderful. She took time to explain my COPD management plan and made sure I understood every step. The nurse on reception, Karen Thompson, was also very welcoming.
>>> 5 stars 

 * sentece: 2: The parking at 45 Albert Street is very limited. I had to walk from the Westfield car park which is difficult at my age.
>>> 2 stars 

 * sentece: 3: I had my appointment on 15 January 2026 at 2pm and wasn't seen until after 3:15pm. Dr James Richardson seemed rushed and didn't listen to my concerns about the medication side effects. I called the clinic at 07 3456 7890 twice before my appointment to ask about preparation and nobody answered. My wife Angela Chen had a much better experience at the Chermside Day Surgery last month.
>>> 1 star 

 * sentece: 4: The physiotherapist Michael was great. He gave me a detailed home exercise program after my knee surgery and followed up via email at susan.obrien82@gmail.com to check on my